# Llama 3.2 1B: Supervised Fine-Tuning and DPO

This notebook explores two stages of post-training a pretrained language model:

1. **Supervised fine-tuning (SFT):** Train on instruction–response examples from Alpaca Cleaned.
2. **Direct Preference Optimization (DPO):** Train on preferred and rejected responses from Orca DPO Pairs.

We compare responses to the same prompts before training, after SFT, and after DPO, keeping generation settings consistent.

The initial SFT run is limited to 100 optimizer updates to check the training pipeline. These short runs and example responses illustrate model behavior; they are not a comprehensive evaluation.

## Environment

The saved run used a Colab GPU runtime with BF16 and TF32 support. Checkpoint files under `/content/` belong to the temporary runtime. Hugging Face access to the Llama model is required.


## Supervised Fine-Tuning

A pretrained base model learns to predict the next token. Supervised fine-tuning continues that training using examples of instructions paired with desired responses.

We use `meta-llama/Llama-3.2-1B` with instruction–response examples from `yahma/alpaca-cleaned`. Each example becomes a conversation containing a user message and an assistant response.

### Load the model and tokenizer

Load the base model and its tokenizer, then configure padding for batched training. The chat template is supplied later when we configure the SFT trainer.

In [1]:
%pip install -q transformers accelerate datasets trl

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Llama-3.2-1B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = "<|finetune_right_pad_id|>"
tokenizer.padding_side = "right"

print("Model and tokenizer loaded.")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model and tokenizer loaded.


### Load and split the instruction dataset

Alpaca Cleaned contains instructions, optional input context, and example responses. We reserve 10% for validation and use the remaining 90% for training.

A fixed random seed makes the split reproducible. Validation examples are excluded from training and used to measure held-out loss.

In [3]:
import datasets
ds = datasets.load_dataset("yahma/alpaca-cleaned")["train"]

split_ds = ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

print(f"SFT Training set size: {len(train_ds)}")
print(f"SFT Evaluation set size: {len(eval_ds)}")

SFT Training set size: 46584
SFT Evaluation set size: 5176


In [4]:
def create_conversation(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": f"{sample['instruction']}\n\n{sample['input']}".strip(),
            },
            {"role": "assistant", "content": sample["output"]},
        ]
    }

train_ds = train_ds.map(
    create_conversation, remove_columns=train_ds.column_names
)
eval_ds = eval_ds.map(
    create_conversation, remove_columns=eval_ds.column_names
)

Map:   0%|          | 0/46584 [00:00<?, ? examples/s]

Map:   0%|          | 0/5176 [00:00<?, ? examples/s]

### Inspect the formatted examples

Check one training example and one validation example. Each should contain a user message with the instruction and any input context, followed by the expected assistant response.

In [5]:
print(train_ds[0])
print()
print(eval_ds[0])

{'messages': [{'content': "Analyze the user's writing style in the given text snippet and provide 3 suggestions to improve it.\n\nWeather now bad outside also it is cold. People no have fun time, always grumble because of cold weather. Use warm clothes protect from freezing.", 'role': 'user'}, {'content': "1. Improve grammar and sentence structure: The sentences are fragmented and lack proper grammar. Consider rewriting them as, 'The weather outside is bad, and it's cold. People are not having a good time, and they are always grumbling because of the cold weather. Wear warm clothes to protect yourself from freezing.' \n2. Use more descriptive words: Replace 'bad' and 'cold' with more specific and descriptive words like 'gloomy' or 'frigid.'\n3. Make the text flow smoothly by using conjunctions and transitions: Use words like 'however,' 'moreover,' and 'therefore' to connect ideas and make the text easier to read.", 'role': 'assistant'}]}

{'messages': [{'content': 'Rearrange the follow

### Configure supervised fine-tuning

The trainer applies Llama’s instruction chat template to the conversation examples and prepares them for next-token training.

We limit this initial run to 100 optimizer updates and sequences of up to 512 tokens. With a batch size of 8 and two gradient accumulation steps, each optimizer update combines up to 16 examples on one GPU.

Every 50 updates, we evaluate validation loss and save a checkpoint. At the end, the trainer restores the checkpoint with the lowest measured validation loss.

In [6]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/Llama-3.2-1B-alpaca-cleaned-sft"

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    chat_template_path="meta-llama/Llama-3.2-1B-Instruct",
    eos_token="<|eot_id|>",
    max_length=512,
    max_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    tf32=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    report_to="none",
    disable_tqdm=False,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/46584 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/46584 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/46584 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/46584 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/5176 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/5176 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/5176 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/5176 [00:00<?, ? examples/s]

### Before fine-tuning

Generate a response from the base model using the chat template configured by the SFT trainer.

We will repeat the same prompt after training with identical generation settings. Greedy decoding selects the highest-scoring token at each step, making the comparison easier to interpret.

In [8]:
messages = [{"role": "user", "content": "Give three tips for staying healthy."}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

model.eval()
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(
    output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
))
model.train();

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three tips for staying healthy.edReader

Give three


In [9]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,1.495250,1.455908,1.433725,0.650202,160452.000000
100,1.332606,1.328105,1.331267,0.668345,312208.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=100, training_loss=1.47511004447937, metrics={'train_runtime': 158.7211, 'train_samples_per_second': 10.081, 'train_steps_per_second': 0.63, 'total_flos': 3669663921831936.0, 'train_loss': 1.47511004447937, 'epoch': 0.03434655675768504})

### After supervised fine-tuning

Repeat the same prompt with identical generation settings. Compare instruction following, relevance, and repetition with the base model’s response.

In [10]:
messages = [{"role": "user", "content": "Give three tips for staying healthy."}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

model.eval()
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(
    output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
))

1. Exercise regularly: Physical activity is essential for staying healthy. Engaging in activities such as jogging, yoga, or swimming can help to keep your heart and lungs healthy, maintain a healthy weight, and reduce the risk of chronic diseases.

2. Get plenty of sleep: Sleep is crucial for staying healthy. Aim for 7-8 hours of sleep per night, and try to avoid sleeping too much or too little.

3. Practice good hygiene: Proper hygiene is essential for staying healthy. Wash your hands frequently with warm water and soap, and avoid touching your face, nose, or mouth without washing your hands first. Additionally, try


### Comparing the responses

Before SFT, the model repeatedly echoed the prompt. After SFT, it produced three relevant tips in a numbered list.

The final sentence is unfinished under the 128-token generation limit. Both responses used the same prompt and decoding settings.

This example shows improved instruction following on one prompt; it does not establish performance across other tasks.

In [11]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/Llama-3.2-1B-alpaca-cleaned-sft/tokenizer_config.json',
 '/content/Llama-3.2-1B-alpaca-cleaned-sft/chat_template.jinja',
 '/content/Llama-3.2-1B-alpaca-cleaned-sft/tokenizer.json')

### Release the SFT training objects

Delete the trainer and model before loading the saved checkpoint for DPO. Garbage collection and clearing unused CUDA cache help release memory.

In [12]:
import gc

del trainer, model
gc.collect()
torch.cuda.empty_cache()

## Direct Preference Optimization

DPO trains a model using pairs of responses to the same prompt: one preferred (`chosen`) and one less preferred (`rejected`).

Starting from our SFT checkpoint, we adjust the model to favor chosen responses over rejected responses relative to a frozen reference model. DPO does not require training a separate reward model or running an online reinforcement-learning loop.

We use preference pairs from `Intel/orca_dpo_pairs`.

### Reload the SFT checkpoint

Load the saved SFT model and tokenizer. This model will be updated during DPO; a separate copy will serve as the frozen reference.

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = OUTPUT_DIR

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float32,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"

print("SFT model and tokenizer reloaded.")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

SFT model and tokenizer reloaded.


### Load the preference dataset

Each example in `Intel/orca_dpo_pairs` contains a prompt, a chosen response, and a rejected response. DPO uses these pairs to learn the dataset’s response preferences.

In [14]:
import datasets

dataset = datasets.load_dataset("Intel/orca_dpo_pairs")["train"]
dataset

README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl: reconstructing file:   0%|          |  0.00B / 36.3MB            

orca_rlhf.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

Dataset({
    features: ['system', 'question', 'chosen', 'rejected'],
    num_rows: 12859
})

### Format the preference pairs

Separate each example into a prompt and two possible assistant responses.

The prompt contains the system message, when present, followed by the user’s question. The chosen and rejected responses remain separate so DPO can compare them.

In [15]:
def to_preference(example):
    prompt = []
    if example["system"]:
        prompt.append({"role": "system", "content": example["system"]})
    prompt.append({"role": "user", "content": example["question"]})

    return {
        "prompt": prompt,
        "chosen": [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
    }

dataset = dataset.map(to_preference, remove_columns=dataset.column_names)
dataset

Map:   0%|          | 0/12859 [00:00<?, ? examples/s]

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 12859
})

### Split the preference dataset

Reserve 10% of the preference pairs for validation. A fixed seed makes the split reproducible.

In [16]:
dpo_split = dataset.train_test_split(test_size=0.1, seed=42)

dpo_train = dpo_split["train"]
dpo_eval = dpo_split["test"]

print(f"DPO training examples: {len(dpo_train):,}")
print(f"DPO validation examples: {len(dpo_eval):,}")

DPO training examples: 11,573
DPO validation examples: 1,286


### Load the reference model

Load a second copy of the saved SFT checkpoint and freeze its parameters. DPO compares the trainable model’s response preferences against this fixed reference.

In [17]:
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float32,
)

ref_model.requires_grad_(False)
ref_model.eval();

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

### Configure DPO training

Run an initial 100 optimizer updates using the training preference pairs. Limit each prompt and response sequence to 512 tokens.

Evaluate on the held-out preference pairs and save a checkpoint every 50 updates. Restore the checkpoint with the lowest validation loss at the end.

The frozen SFT reference stays unchanged throughout training.

In [18]:
from trl import DPOTrainer, DPOConfig

DPO_DIR = "/content/Llama-3.2-1B-alpaca-cleaned-orca-dpo"

dpo_config = DPOConfig(
    output_dir=DPO_DIR,
    max_length=512,
    max_steps=100,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=1e-6,
    beta=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    fp16=False,
    tf32=True,
    seed=42,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/11573 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/11573 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1286 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/1286 [00:00<?, ? examples/s]

### Train with preference pairs

Train for 100 optimizer updates while keeping the reference model frozen. Evaluate on held-out preference pairs and save checkpoints every 50 updates.

DPO loss measures the preference-training objective; it is not directly comparable to SFT’s next-token loss.

In [19]:
dpo_trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.057696,0.061847,1.798285,548800.000000,0.999910,1.052235,0.595304,-0.721945,-6.336568,0.989362,5.614623,-236.725114,-324.118029
100,0.038945,0.052318,1.830329,1103130.000000,0.995374,1.046606,0.593503,-0.861946,-7.192781,0.989362,6.330835,-238.125128,-332.680164


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=100, training_loss=0.11179856389760971, metrics={'train_runtime': 216.7257, 'train_samples_per_second': 7.383, 'train_steps_per_second': 0.461, 'total_flos': 9229956110057472.0, 'train_loss': 0.11179856389760971, 'epoch': 0.15631105900742479})

### After DPO

Repeat the same prompt and generation settings used before and after SFT. Compare instruction following, relevance, and repetition across the three responses.

A change in this single response does not establish an overall improvement.

In [20]:
messages = [{"role": "user", "content": "Give three tips for staying healthy."}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

model.eval()
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(
    output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
))

1. Exercise regularly: Physical activity is essential for staying healthy. Engaging in activities such as jogging, yoga, or swimming can help to keep your heart and lungs healthy, maintain a healthy weight, and reduce stress and anxiety.

2. Get plenty of sleep: Sleep is crucial for staying healthy. Aim for 7-8 hours of sleep per night, and try to avoid sleeping too much or too little.

3. Practice good hygiene: Proper hygiene is essential for staying healthy. Wash your hands frequently, use hand sanitizer, and avoid touching your face after handling raw meats or poultry.


In [29]:
dpo_trainer.save_model(DPO_DIR)
tokenizer.save_pretrained(DPO_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/Llama-3.2-1B-alpaca-cleaned-orca-dpo/tokenizer_config.json',
 '/content/Llama-3.2-1B-alpaca-cleaned-orca-dpo/chat_template.jinja',
 '/content/Llama-3.2-1B-alpaca-cleaned-orca-dpo/tokenizer.json')

## Results

The base model repeated the example prompt. After SFT, the same prompt produced a relevant numbered response, with its final sentence cut short by the generation limit. DPO produced a similar response with wording changes.

Both training stages completed 100 updates with held-out validation. These results demonstrate the post-training workflow; the single-prompt comparison does not establish an overall advantage for DPO.
